# 🔍 UYDS SSS Veri Ayıklama Pipeline v3.1
## Standart+UYDS Format · Model Hash Checkpoint · Opsiyonel Hunspell

**v3.1 final:**
- 🔴 Standart formatta `_konu` KeyError giderildi (dinamik kolon seçimi)
- 🔴 Checkpoint model/config hash doğrulaması
- 🟠 Hunspell opsiyonel (VS Code'da çökme yok)
- 🟠 Semantic search `top_k` genişletildi (kaçırma riski az)
- 🟠 Doğrulama seti indeks + kayıt hash çift koruma
- 🟡 Colab'da pandas/numpy/matplotlib sabitlenmez
---

## ⚙️ Hücre 0: Kurulum + Tüm İmportlar + Config + Checkpoint

In [ ]:
import sys, os, warnings, pickle, json, re, unicodedata, gc, time, hashlib
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

COLAB_MI = "google.colab" in sys.modules
print(f"🖥️ {'Colab' if COLAB_MI else 'VS Code'}")

if COLAB_MI:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null
    if gpu_info: print(f"✅ GPU: {gpu_info[0]}")
    else: print("⚠️ GPU yok, CPU kullanılacak.")
    # pandas/numpy/matplotlib sabitlenmez (Colab'da zaten var)
    !apt-get update -qq && apt-get install -y -qq build-essential pkg-config libhunspell-dev hunspell-tr
    !pip install -qU sentence-transformers==3.4.1 transformers==4.47.1 openpyxl==3.1.5 scikit-learn==1.6.0 lingua-language-detector==2.0.2 hunspell==0.5.5 tqdm==4.67.1 pyyaml

import glob as _glob
# --- Config ---
import yaml
CFG = {}
config_adaylari = [os.path.join(os.getcwd(),"config","config.yaml"), "config/config.yaml", os.path.join(os.getcwd(),"..","config","config.yaml")]
if COLAB_MI:
    config_adaylari.extend(_glob.glob("/content/drive/MyDrive/**/google_collab_egitim/config/config.yaml", recursive=True))
for aday in config_adaylari:
    if os.path.exists(aday):
        with open(aday,"r",encoding="utf-8") as f: CFG = yaml.safe_load(f) or {}
        print(f"✅ Config: {aday}"); break
if not CFG: print("⚠️ config.yaml bulunamadı, varsayılanlar kullanılacak.")

def cfg(*keys, default=None):
    d = CFG
    for k in keys: d = d.get(k) if isinstance(d,dict) else None
    return d if d is not None else default

# --- Dizinler ---
if COLAB_MI:
    CALISMA_DIZINI = "/content/drive/MyDrive"
    adaylar = _glob.glob("/content/drive/MyDrive/**/google_collab_egitim", recursive=True)
    if adaylar: CALISMA_DIZINI = adaylar[0]
else:
    CALISMA_DIZINI = os.getcwd()

EXCEL_ADAY = [os.path.join(CALISMA_DIZINI,"data","input","UYDS_SSS_Listesi.xlsx"), os.path.join(CALISMA_DIZINI,"UYDS_SSS_Listesi.xlsx")]
EXCEL_DOSYASI = next((p for p in EXCEL_ADAY if os.path.exists(p)), None)
if EXCEL_DOSYASI is None: raise FileNotFoundError(f"❌ Excel bulunamadı: {EXCEL_ADAY}")

from datetime import datetime
zaman_damgasi = datetime.now().strftime("%Y-%m-%d_%H%M")
CIKTI_DIZINI = os.path.join(CALISMA_DIZINI, "data", "output", zaman_damgasi)
os.makedirs(CIKTI_DIZINI, exist_ok=True)

DOGRULAMA_DIZINI = os.path.join(CALISMA_DIZINI, "data", "validation")
os.makedirs(DOGRULAMA_DIZINI, exist_ok=True)
dogrulama_yolu = os.path.join(DOGRULAMA_DIZINI, "dogrulama_seti.xlsx")

CKPT_DIZINI = os.path.join(CALISMA_DIZINI, "checkpoints", "latest")
os.makedirs(CKPT_DIZINI, exist_ok=True)

print(f"📄 Excel: {EXCEL_DOSYASI}")
print(f"📁 Çıktı: {CIKTI_DIZINI}")
print(f"🔖 Doğrulama: {dogrulama_yolu}")
print(f"💾 Checkpoint: {CKPT_DIZINI}")

# --- Hash yardımcıları ---
def dosya_hashi(yol):
    h = hashlib.sha256()
    with open(yol, "rb") as f:
        while parca := f.read(1024*1024): h.update(parca)
    return h.hexdigest()

def metin_hashi(metin):
    return hashlib.sha256(str(metin).encode("utf-8")).hexdigest()

KAYNAK_EXCEL_HASHI = dosya_hashi(EXCEL_DOSYASI)

def dataframe_ckpt_kaydet(ad, df):
    try: yol = os.path.join(CKPT_DIZINI, f"{ad}.pkl"); df.to_pickle(yol); print(f"💾 Checkpoint: {yol}")
    except Exception as e: print(f"⚠️ Checkpoint hatası: {e}")

def dizi_ckpt_kaydet(ad, dizi):
    try: np.save(os.path.join(CKPT_DIZINI, f"{ad}.npy"), dizi)
    except Exception as e: print(f"⚠️ Checkpoint hatası: {e}")

# --- Checkpoint devam kontrolü (Excel + Model + Config hash) ---
VERI_CKPT = os.path.join(CKPT_DIZINI, "veri_h8.pkl")
VEKTOR_CKPT = os.path.join(CKPT_DIZINI, "tum_vektorler.npy")
INDERS_CKPT = os.path.join(CKPT_DIZINI, "gecerli_indeksler.pkl")
META_CKPT = os.path.join(CKPT_DIZINI, "meta.json")

DEVAM_ET = False
MODEL_ADI = cfg("model","adi",default="sentence-transformers/paraphrase-multilingual-mpnet-base-v2")
CONFIG_HASH = metin_hashi(json.dumps(CFG, sort_keys=True, ensure_ascii=False)) if CFG else ""
KURAL_SETI_SURUMU = "2026-08-04.2"

if all(os.path.exists(p) for p in [VERI_CKPT, VEKTOR_CKPT, INDERS_CKPT, META_CKPT]):
    try:
        with open(META_CKPT, "r", encoding="utf-8") as f: meta = json.load(f)
        guncel_excel_hash = dosya_hashi(EXCEL_DOSYASI)
        if (meta.get("excel_hash") == guncel_excel_hash
            and meta.get("model") == MODEL_ADI
            and meta.get("config_hash") == CONFIG_HASH
            and meta.get("kural_seti_surumu") == KURAL_SETI_SURUMU):
            DEVAM_ET = True
            print(f"\n🔄 Uyumlu checkpoint bulundu (Excel + Model + Config eşleşti).")
            print("   Notebook'u Run All ile çalıştırabilirsiniz.")
        else:
            farklar = []
            if meta.get("excel_hash") != guncel_excel_hash: farklar.append("Excel")
            if meta.get("model") != MODEL_ADI: farklar.append("Model")
            if meta.get("config_hash") != CONFIG_HASH: farklar.append("Config")
            if meta.get("kural_seti_surumu") != KURAL_SETI_SURUMU: farklar.append("Kural seti")
            print(f"\n⚠️ Checkpoint geçersiz ({', '.join(farklar)} değişmiş). Sıfırdan başlanacak.")
    except Exception as e:
        print(f"⚠️ Checkpoint meta okunamadı: {e}")

if not DEVAM_ET:
    print("\nℹ️ Geçerli checkpoint yok, normal akışla devam.")

print("✅ Hücre 0 tamam.")

## 📖 Hücre 1: Excel Oku + Sütun Eşleştir

In [ ]:
if DEVAM_ET:
    print("⏭️ Checkpoint var, Hücre 1 atlandı.")
else:
    def adi_normalize_et(ad):
        if not isinstance(ad, str): return ""
        ad = unicodedata.normalize("NFKC", ad).casefold()
        ad = ad.replace("ı","i")
        ad = unicodedata.normalize("NFKD", ad)
        return "".join(karakter for karakter in ad if not unicodedata.combining(karakter)).strip()
    ham_veri = pd.read_excel(EXCEL_DOSYASI, sheet_name=0, header=None, engine="openpyxl", dtype=object)
    bilinen_basliklar = {"modul", "musteri", "madde tarihi", "madde no", "konu", "aciklama", "yapilacak", "yapilan islem", "kategori", "soru", "cevap", "kaynak"}
    baslik_satiri = next((i for i, satir in ham_veri.head(20).iterrows() if len({adi_normalize_et(deger) for deger in satir.tolist()} & bilinen_basliklar) >= 3), None)
    if baslik_satiri is None:
        raise ValueError("Excel başlık satırı otomatik olarak bulunamadı.")
    veri = pd.read_excel(EXCEL_DOSYASI, sheet_name=0, header=baslik_satiri, engine="openpyxl", dtype=object)
    veri = veri.dropna(axis=1, how="all")
    print(f"✅ Başlık satırı: {baslik_satiri + 1}. satır")
    print(f"✅ {len(veri)} satır × {len(veri.columns)} sütun: {list(veri.columns)}")
    mk = {adi_normalize_et(k): k for k in veri.columns}
    H = {"modul":["modul"],"konu":["konu"],"aciklama":["aciklama"],"yapilacak":["yapilacak"],"yapilan_islem":["yapilan islem","yapilan_islem"],"madde_no":["madde no","madde_no","harici kod"],"madde_tarihi":["madde tarihi","madde_tarihi"],"musteri":["musteri"]}
    S = {"kategori":["kategori"],"soru":["soru"],"cevap":["cevap"],"kaynak":["kaynak dokuman","kaynak belge","kaynak"]}
    def kle(hedef):
        e={}
        for a,ads in hedef.items():
            for ad in ads:
                if adi_normalize_et(ad) in mk: e[a]=mk[adi_normalize_et(ad)]; break
        return e
    ue,se=kle(H),kle(S)
    ft,esleme = ("uyds",ue) if len(ue)>=4 else (("standart",se) if len(se)>=3 else (None,None))
    if ft is None: raise ValueError(f"Format tanınamadı: {list(mk.keys())}")
    print(f"✅ Format: {ft}. Eşlenen: {list(esleme.keys())}")
    def mtd(d):
        if d is None or (isinstance(d,float) and pd.isna(d)): return ""
        if isinstance(d,(int,float)): return str(int(d)) if d==int(d) else str(d)
        return str(d).strip()
    print("✅ Hücre 1 tamam.")

## 📝 Hücre 2: Kayıt Metni + Hash Tam Tekrar + Standart Format Düzeltmesi

In [ ]:
if DEVAM_ET:
    print("⏭️ Checkpoint var, Hücre 2 atlandı.")
else:
    # 🔴 Standart formatta _konu ve _madde_no oluştur (doğrulama seti için)
    if ft=="uyds":
        for a in ["modul","konu","aciklama","yapilacak","yapilan_islem","madde_no","madde_tarihi","musteri"]:
            veri[f"_{a}"] = veri[esleme[a]].apply(mtd) if a in esleme else ""
    else:
        veri["_kategori_1"]=veri[esleme["kategori"]].apply(mtd)
        veri["_soru_1"]=veri[esleme["soru"]].apply(mtd)
        veri["_cevap_1"]=veri[esleme["cevap"]].apply(mtd)
        veri["_modul"]=veri["_kategori_1"]
        veri["_konu"]=veri["_soru_1"]
        veri["_madde_no"]=""

    def kmo(satir):
        p=[]
        if ft=="uyds":
            if satir.get("_modul"): p.append(f"[{satir['_modul']}]")
            if satir.get("_konu"): p.append(satir["_konu"])
            if satir.get("_aciklama"): p.append(satir["_aciklama"])
            if satir.get("_yapilacak"): p.append(f"Yapılacak: {satir['_yapilacak']}")
            if satir.get("_yapilan_islem"): p.append(f"Yapılan İşlem: {satir['_yapilan_islem']}")
        else:
            if satir.get("_kategori_1"): p.append(f"[{satir['_kategori_1']}]")
            if satir.get("_soru_1"): p.append(satir["_soru_1"])
            if satir.get("_cevap_1"): p.append(satir["_cevap_1"])
        return ". ".join(p) if p else ""

    veri["_kayit_metni"]=veri.apply(kmo,axis=1)
    veri["_kategori"]=veri["_modul"]

    # Hash tabanlı tam tekrar
    def tekrar_icin_normalize(metin):
        if not metin: return ""
        m = unicodedata.normalize("NFKC", str(metin)); m = m.casefold(); m = re.sub(r"\s+", " ", m).strip()
        return m
    ilk_gorulen = {}
    veri["_tam_tekrar_indeksi"] = -1
    for idx in veri.index:
        normal = tekrar_icin_normalize(veri.at[idx, "_kayit_metni"])
        if not normal: continue
        if normal in ilk_gorulen: veri.at[idx, "_tam_tekrar_indeksi"] = ilk_gorulen[normal]
        else: ilk_gorulen[normal] = idx
    veri["_hata_tam_tekrar"] = veri["_tam_tekrar_indeksi"] >= 0
    print(f"📊 {len(veri)} satır, {veri['_kategori'].nunique()} kategori, 🔴 Hash tam tekrar: {veri['_hata_tam_tekrar'].sum()}")
    print("✅ Hücre 2 tamam.")

## 📊 Hücre 3: Doğrulama Seti (dinamik kolon + indeks + kayıt hash)

In [ ]:
if DEVAM_ET:
    print("⏭️ Checkpoint var, Hücre 3 atlandı.")
else:
    DB = cfg("dogrulama","boyut",default=200)
    DOGRULAMA_BOYUTU = min(DB, len(veri))
    rs = cfg("dogrulama","random_state",default=42)
    veri["_manuel_etiket"]=""
    kategori_sayisi = max(veri["_kategori"].nunique(dropna=False) if "_kategori" in veri.columns else 1, 1)
    kategori_basina = max(1, DOGRULAMA_BOYUTU // kategori_sayisi)
    parcalar = []
    if "_kategori" in veri.columns:
        for _, grup in veri.groupby("_kategori", dropna=False):
            adet = min(kategori_basina, len(grup))
            if adet:
                parcalar.append(grup.sample(n=adet, random_state=rs))
    else:
        parcalar.append(veri.sample(n=min(DOGRULAMA_BOYUTU, len(veri)), random_state=rs))
    dogrulama_verisi = pd.concat(parcalar) if parcalar else veri.iloc[0:0].copy()
    eksik = DOGRULAMA_BOYUTU - len(dogrulama_verisi)
    if eksik > 0:
        kalan = veri.drop(index=dogrulama_verisi.index)
        if len(kalan) > 0:
            ek_ornek = kalan.sample(n=min(eksik, len(kalan)), random_state=rs)
            dogrulama_verisi = pd.concat([dogrulama_verisi, ek_ornek])
    if len(dogrulama_verisi) > DOGRULAMA_BOYUTU:
        dogrulama_verisi = dogrulama_verisi.sample(n=DOGRULAMA_BOYUTU, random_state=rs)
    dogrulama_indeksleri = dogrulama_verisi.index
    veri.loc[dogrulama_indeksleri,"_manuel_etiket"]="bekliyor"

    # 🔴 Dinamik kolon seçimi (standart formatta _konu olmayabilir)
    dogrulama_kolonlari = ["_modul", "_konu", "_kayit_metni", "_madde_no", "_manuel_etiket"]
    dogrulama_kolonlari = [k for k in dogrulama_kolonlari if k in veri.columns]
    ds = veri.loc[dogrulama_indeksleri, dogrulama_kolonlari].copy()
    ds.insert(0, "_veri_indeksi", dogrulama_indeksleri)
    # 🔴 Kayıt hash'i (indeks geçersiz olsa bile doğrulama için)
    ds["_kayit_hash"] = ds["_kayit_metni"].apply(lambda x: hashlib.sha256(tekrar_icin_normalize(x).encode("utf-8")).hexdigest() if x else "")
    ds["_kaynak_excel_hash"] = KAYNAK_EXCEL_HASHI

    if os.path.exists(dogrulama_yolu):
        mevcut_ds = pd.read_excel(dogrulama_yolu, engine="openpyxl")
        eski_hash = mevcut_ds["_kaynak_excel_hash"].dropna().iloc[0] if "_kaynak_excel_hash" in mevcut_ds.columns and mevcut_ds["_kaynak_excel_hash"].notna().any() else None
        if eski_hash != KAYNAK_EXCEL_HASHI:
            print("⚠️ Doğrulama seti farklı kaynak Excel'e ait, mevcut dosya korunuyor.")
        else:
            print(f"⚠️ Mevcut doğrulama seti KORUNDU: {dogrulama_yolu}")
    else:
        ds.to_excel(dogrulama_yolu, index=False, engine="openpyxl")
        print(f"📋 Doğrulama seti: {dogrulama_yolu} ({DOGRULAMA_BOYUTU} satır)")
    print("✅ Hücre 3 tamam.")

## 🔍 Hücre 4: Yapısal Kontroller

In [ ]:
if DEVAM_ET:
    print("⏭️ Checkpoint var, Hücre 4 atlandı.")
else:
    SADECE_SEMBOL_RE = re.compile(r"^[\d\W_]+$")
    def ts(m): return len(str(m).split()) if m else 0
    def ssm(m): return bool(SADECE_SEMBOL_RE.match(str(m))) if m else False
    def kc(m): k=str(m).lower().split(); return len(set(k))/len(k) if k else 0.0
    KISA=cfg("yapisal","kisa_metin_esik",default=5); UZUN=cfg("yapisal","uzun_metin_esik",default=300)
    TTR_ESIK=cfg("yapisal","dusuk_ttr_esik",default=0.25); TTR_MIN=cfg("yapisal","dusuk_ttr_min_kelime",default=10)
    hata_kolonlari=["_hata_tam_tekrar"]
    veri["_token_sayisi"]=veri["_kayit_metni"].apply(ts)
    veri["_hata_bos"]=veri["_kayit_metni"]==""; hata_kolonlari.append("_hata_bos")
    veri["_hata_kisa"]=(veri["_token_sayisi"]>0)&(veri["_token_sayisi"]<KISA); hata_kolonlari.append("_hata_kisa")
    veri["_hata_uzun"]=veri["_token_sayisi"]>UZUN; hata_kolonlari.append("_hata_uzun")
    veri["_hata_sembol"]=veri["_kayit_metni"].apply(ssm); hata_kolonlari.append("_hata_sembol")
    veri["_ttr"]=veri["_kayit_metni"].apply(kc)
    veri["_hata_dusuk_ttr"]=(veri["_token_sayisi"]>TTR_MIN)&(veri["_ttr"]<TTR_ESIK); hata_kolonlari.append("_hata_dusuk_ttr")
    for k in hata_kolonlari:
        s=veri[k].sum()
        if s>0: print(f"   {k}: {s}")
    print("✅ Hücre 4 tamam.")

## 🌐 Hücre 5: Dil Tespiti

In [ ]:
if DEVAM_ET:
    print("⏭️ Checkpoint var, Hücre 5 atlandı.")
else:
    from lingua import Language, LanguageDetectorBuilder
    DMK=cfg("dil","min_kelime",default=8)
    lds=[Language.TURKISH,Language.ENGLISH,Language.ARABIC,Language.RUSSIAN,Language.FRENCH,Language.GERMAN]
    detector=LanguageDetectorBuilder.from_languages(*lds).build()
    def dte(metin,ts):
        if not metin or not str(metin).strip(): return "bos"
        if ts<DMK: return "belirsiz"
        try: s=detector.detect_language_of(str(metin)[:300]); return s.iso_code_639_1.name.lower() if s else "belirsiz"
        except: return "belirsiz"
    veri["_dil"]=veri.apply(lambda r: dte(r["_kayit_metni"],r["_token_sayisi"]),axis=1)
    veri["_hata_dil"]=~veri["_dil"].isin(["tr","turkish","bos","belirsiz"]); hata_kolonlari.append("_hata_dil")
    print(f"📊 {veri['_dil'].value_counts().to_dict()}")
    print("✅ Hücre 5 tamam.")

## 📝 Hücre 6: Yazım Denetimi (🟠 opsiyonel Hunspell)

In [ ]:
YE=cfg("yazim","esik",default=0.35); YMK=cfg("yazim","min_kelime",default=10)
HUNSPELL_VAR = False
try:
    import hunspell
    hobj=None
    for yol in ["/usr/share/hunspell/tr_TR","/usr/share/myspell/tr_TR"]:
        try: hobj=hunspell.HunSpell(yol+".dic",yol+".aff"); HUNSPELL_VAR=True; print(f"✅ Hunspell: {yol}"); break
        except: pass
    if not HUNSPELL_VAR: print("⚠️ Hunspell sözlük bulunamadı.")
except ImportError:
    print("⚠️ Hunspell paketi bulunamadı (VS Code/Windows). Yazım denetimi atlanacak.")

if HUNSPELL_VAR and not DEVAM_ET:
    def yo(metin):
        if not metin or hobj is None: return 0.0
        klm=re.findall(r"\b[a-zA-ZçğıöşüÇĞİÖŞÜ]{3,}\b",str(metin).lower())
        if len(klm)<YMK: return 0.0
        return sum(1 for k in klm if not hobj.spell(k))/len(klm)
    veri["_yazim_orani"]=veri["_kayit_metni"].apply(yo)
    veri["_hata_yazim"]=veri["_yazim_orani"]>YE
else:
    if not DEVAM_ET:
        veri["_yazim_orani"]=0.0; veri["_hata_yazim"]=False

if not DEVAM_ET:
    hata_kolonlari.append("_hata_yazim")
    print(f"📝 Yazım hatası: {veri['_hata_yazim'].sum()}")
else:
    print("⏭️ Checkpoint var, Hücre 6 atlandı.")
print("✅ Hücre 6 tamam.")

## 🧠 Hücre 7: Embedding Modeli

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
device="cuda" if torch.cuda.is_available() else "cpu"
BS=cfg("model","batch_size",default=64)

if DEVAM_ET:
    print(f"🔄 Checkpoint'ten devam — model yeniden yükleniyor...")
    try:
        model = SentenceTransformer(MODEL_ADI, device=device, trust_remote_code=False)
    except Exception as hata:
        raise RuntimeError(f"Model yüklenemedi: {MODEL_ADI}. İnternet bağlantısını veya Hugging Face cache'ini kontrol et.") from hata
    if device=="cuda" and cfg("model","fp16",default=True): model.half()
    boyut = model.get_sentence_embedding_dimension()
    print(f"✅ Model: {MODEL_ADI}, boyut: {boyut}")

    veri = pd.read_pickle(VERI_CKPT)
    tum_vektorler = np.load(VEKTOR_CKPT)
    with open(INDERS_CKPT, "rb") as f: gecerli_indeksler = pickle.load(f)
    gm = veri["_kayit_metni"].str.strip()!=""
    metinler = veri.loc[gm,"_kayit_metni"].tolist()

    hata_kolonlari = [k for k in veri.columns if k.startswith("_hata_")]
    print(f"✅ Checkpoint yüklendi: {len(veri)} satır, {tum_vektorler.shape} vektör, {len(hata_kolonlari)} hata kolonu")
    print("✅ Hücre 7 (devam) tamam.")
else:
    print(f"🖥️ {device}" + (f" — {torch.cuda.get_device_name(0)}" if device=="cuda" else ""))
    try:
        model = SentenceTransformer(MODEL_ADI, device=device, trust_remote_code=False)
    except Exception as hata:
        raise RuntimeError(f"Model yüklenemedi: {MODEL_ADI}. İnternet bağlantısını veya Hugging Face cache'ini kontrol et.") from hata
    if device=="cuda" and cfg("model","fp16",default=True): model.half(); print("   ✅ FP16")
    boyut = model.get_sentence_embedding_dimension()
    print(f"✅ Boyut: {boyut}")
    print("✅ Hücre 7 tamam.")

## 🔢 Hücre 8: Embedding Üret + Checkpoint Kaydet

In [ ]:
if DEVAM_ET:
    print("⏭️ Checkpoint var, Hücre 8 atlandı (veriler Hücre 7'de yüklendi).")
else:
    gm = veri["_kayit_metni"].str.strip()!=""
    metinler = veri.loc[gm,"_kayit_metni"].tolist()
    gecerli_indeksler = veri[gm].index.tolist()
    if len(metinler) == 0: raise ValueError("❌ Embedding üretilecek geçerli metin bulunamadı.")

    print(f"🔢 Embedding: {len(metinler)} satır")
    t0=time.time()
    tum_vektorler = model.encode(metinler, batch_size=BS, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    print(f"   {tum_vektorler.shape}, {time.time()-t0:.1f}s, {tum_vektorler.nbytes/1e6:.1f} MB")
    veri["_vektor_indeksi"]=-1
    for i,idx in enumerate(gecerli_indeksler): veri.at[idx,"_vektor_indeksi"]=i

    dataframe_ckpt_kaydet("veri_h8", veri)
    dizi_ckpt_kaydet("tum_vektorler", tum_vektorler)
    with open(os.path.join(CKPT_DIZINI, "gecerli_indeksler.pkl"), "wb") as f: pickle.dump(gecerli_indeksler, f)
    # 🔴 Excel + Model + Config hash
    checkpoint_meta = {
        "excel_yolu": os.path.abspath(EXCEL_DOSYASI), "excel_hash": dosya_hashi(EXCEL_DOSYASI),
        "model": MODEL_ADI, "config_hash": CONFIG_HASH, "kural_seti_surumu": KURAL_SETI_SURUMU
    }
    with open(os.path.join(CKPT_DIZINI, "meta.json"), "w", encoding="utf-8") as f: json.dump(checkpoint_meta, f, ensure_ascii=False, indent=2)
    print("💾 Checkpoint: veri_h8.pkl + tum_vektorler.npy + meta.json (Excel+Model+Config hash)")
    print("✅ Hücre 8 tamam.")

## 🔁 Hücre 9: Semantic Search (🟠 genişletilmiş top_k)

In [ ]:
from sentence_transformers.util import semantic_search, dot_score
GA = cfg("benzerlik","guclu_aday_esik",default=0.95)
IE = cfg("benzerlik","incele_esik",default=0.88)
TK = cfg("benzerlik","top_k",default=10)
n = len(tum_vektorler)

# 🟠 Genişletilmiş top_k (kaçırma riskini azaltır)
ARAMA_TOP_K = min(max(TK * 5, 50), n)
print(f"🔍 Semantic search: {n} kayıt, top_k={ARAMA_TOP_K}...")
t0=time.time()
arama_sonuclari = semantic_search(tum_vektorler, tum_vektorler, top_k=ARAMA_TOP_K, score_function=dot_score)
print(f"   {time.time()-t0:.1f}s")

veri["_benzer_indeksi"]=-1; veri["_benzer_skor"]=0.0; veri["_benzer_madde_no"]=""; veri["_benzer_metin"]=""
veri["_benzer_seviye"]="temiz"
ga_c=0; ic_c=0

for i, sonuclar in enumerate(arama_sonuclari):
    idx = gecerli_indeksler[i]
    if veri.at[idx,"_hata_tam_tekrar"]: continue
    eyj=-1; eys=-1.0
    for es in sonuclar:
        j=es["corpus_id"]; sk=float(es["score"])
        if j>=i: continue
        if veri.at[gecerli_indeksler[j],"_hata_tam_tekrar"]: continue
        if sk>eys: eys=sk; eyj=j
    if eyj>=0 and eys>0:
        eidx=gecerli_indeksler[eyj]; em=str(veri.at[eidx,"_kayit_metni"])
        veri.at[idx,"_benzer_indeksi"]=eidx; veri.at[idx,"_benzer_skor"]=eys
        veri.at[idx,"_benzer_madde_no"]=str(veri.at[eidx,"_madde_no"]); veri.at[idx,"_benzer_metin"]=em[:200]
        if eys>=GA: veri.at[idx,"_benzer_seviye"]="guclu_aday"; ga_c+=1
        elif eys>=IE: veri.at[idx,"_benzer_seviye"]="incele"; ic_c+=1

veri["_hata_benzer_guclu"]=veri["_benzer_seviye"]=="guclu_aday"
veri["_hata_benzer_incele"]=veri["_benzer_seviye"]=="incele"
hata_kolonlari.extend(["_hata_benzer_guclu","_hata_benzer_incele"])
print(f"📊 Güçlü aday: {ga_c} | İncele: {ic_c}")

del arama_sonuclari; gc.collect()
if device=="cuda": torch.cuda.empty_cache()
benzerlik_skorlari = veri.loc[veri["_benzer_skor"]>0,"_benzer_skor"].values
print("✅ Hücre 9 tamam.")

## 🎯 Hücre 10: Outlier (AND mantığı)

In [ ]:
from sklearn.cluster import DBSCAN; from sklearn.ensemble import IsolationForest
DE = cfg("outlier","dbscan_eps",default=0.35); DMS = cfg("outlier","dbscan_min_samples",default=5)

if len(tum_vektorler) < 5:
    print("⚠️ Kayıt < 5, outlier atlandı.")
    outlier_dbscan = np.zeros(len(tum_vektorler), dtype=bool); outlier_iso = np.zeros(len(tum_vektorler), dtype=bool)
else:
    print("🔍 DBSCAN...")
    outlier_dbscan = DBSCAN(eps=DE, min_samples=DMS, metric="cosine", n_jobs=-1).fit_predict(tum_vektorler)==-1
    print(f"   DBSCAN outlier: {outlier_dbscan.sum()}")
    if n<500: cv=0.02
    elif n<2000: cv=0.03
    elif n<10000: cv=0.04
    else: cv="auto"
    print(f"🌲 IsoForest (contamination={cv})...")
    outlier_iso = IsolationForest(contamination=cv, random_state=42, n_jobs=-1).fit_predict(tum_vektorler)==-1
    print(f"   IsoForest outlier: {outlier_iso.sum()}")

veri["_outlier_dbscan"]=False; veri["_outlier_iso"]=False
for i,idx in enumerate(gecerli_indeksler):
    veri.at[idx,"_outlier_dbscan"]=bool(outlier_dbscan[i]); veri.at[idx,"_outlier_iso"]=bool(outlier_iso[i])
veri["_hata_outlier"] = veri["_outlier_dbscan"] & veri["_outlier_iso"]
hata_kolonlari.append("_hata_outlier")
hata_kolonlari.extend(["_outlier_dbscan","_outlier_iso"])
print(f"📊 Outlier (AND): {veri['_hata_outlier'].sum()} (DBSCAN: {outlier_dbscan.sum()}, IF: {outlier_iso.sum()})")
gc.collect()
if device=="cuda": torch.cuda.empty_cache()
print("✅ Hücre 10 tamam.")

## 🏷️ Hücre 11: Kategori Centroid (LOO)

In [ ]:
from collections import defaultdict
KE=cfg("kategori","uyumsuzluk_esik",default=0.25); KMK=cfg("kategori","min_kayit",default=4)
kategori_vektorleri = defaultdict(list)
for i,idx in enumerate(gecerli_indeksler):
    kat=str(veri.at[idx,"_kategori"])
    if kat and kat not in ("nan","Belirtilmemiş"): kategori_vektorleri[kat].append(i)
print(f"🏷️ Kategori: {len(kategori_vektorleri)} (≥{KMK} centroid alır)")
veri["_kategori_cosine"]=-1.0; veri["_hata_kategori"]=False; uyumsuz=0
for kat, v_indeksler in kategori_vektorleri.items():
    if len(v_indeksler)<KMK: continue
    vektorler=tum_vektorler[v_indeksler]; toplam=vektorler.sum(axis=0); n_kat=len(v_indeksler)
    for yi,gi in enumerate(v_indeksler):
        centroid=(toplam-tum_vektorler[gi])/(n_kat-1); centroid=centroid/(np.linalg.norm(centroid)+1e-8)
        cos=float(np.dot(tum_vektorler[gi],centroid))
        idx=gecerli_indeksler[gi]; veri.at[idx,"_kategori_cosine"]=cos
        if cos<KE: veri.at[idx,"_hata_kategori"]=True; uyumsuz+=1
hata_kolonlari.append("_hata_kategori")
print(f"📊 Kategori uyumsuz (LOO): {uyumsuz}")
print("✅ Hücre 11 tamam.")

## 📋 Hücre 12: Hata Konsolidasyonu

In [ ]:
HATA_SINIFI_ETIKETLERI = {
    "_hata_bos":("Boş kayıt","OTOMATİK_SİL"),"_hata_sembol":("Sadece sayı/sembol","OTOMATİK_SİL"),
    "_hata_tam_tekrar":("Hash tam metin tekrarı","OTOMATİK_SİL"),
    "_hata_benzer_guclu":("Çok yüksek benzerlik","GÜÇLÜ_ADAY"),
    "_hata_benzer_incele":("Yüksek benzerlik","İNCELE"),"_hata_kisa":("Çok kısa metin","İNCELE"),
    "_hata_uzun":("Çok uzun metin","İNCELE"),"_hata_dusuk_ttr":("Düşük bilgi yoğunluğu","İNCELE"),
    "_hata_dil":("Türkçe dışı içerik","İNCELE"),"_hata_yazim":("Yazım hatası yoğun","İNCELE"),
    "_hata_outlier":("Outlier (DBSCAN AND IF)","İNCELE"),
    "_hata_kategori":("Kategori uyumsuz","İNCELE"),
}
AKSIYON_ONCELIGI = {"OTOMATİK_SİL":4,"GÜÇLÜ_ADAY":3,"İNCELE":2,"TEMİZ":0}
veri["_hata_siniflari"]=""; veri["_oneri_aksiyon"]="TEMİZ"
for idx in veri.index:
    sf=[]; ek="TEMİZ"
    for k,(et,ak) in HATA_SINIFI_ETIKETLERI.items():
        if k in veri.columns and veri.at[idx,k]:
            sf.append(et)
            if AKSIYON_ONCELIGI[ak]>AKSIYON_ONCELIGI[ek]: ek=ak
    veri.at[idx,"_hata_siniflari"]="; ".join(sf) if sf else ""
    veri.at[idx,"_oneri_aksiyon"]=ek
aksiyon_dagilimi = veri["_oneri_aksiyon"].value_counts()
temiz_satir = (veri["_oneri_aksiyon"]=="TEMİZ").sum()
hatali_satir = len(veri)-temiz_satir
print("="*50); print("📊 KONSOLİDASYON"); print("="*50)
for k,(et,_) in HATA_SINIFI_ETIKETLERI.items():
    s=veri[k].sum() if k in veri.columns else 0
    if s>0: print(f"  {et}: {s}")
print(f"\n📊 Aksiyon: {aksiyon_dagilimi.to_dict()}")
print(f"📊 {len(veri)} toplam | {hatali_satir} hatalı | {temiz_satir} temiz")
print("✅ Hücre 12 tamam.")

## 🎯 Hücre 13: Eşik Kalibrasyonu (indeks + kayıt hash)

In [ ]:
if os.path.exists(dogrulama_yolu):
    ev=pd.read_excel(dogrulama_yolu,engine="openpyxl")
    eski_hashler = ev["_kaynak_excel_hash"].dropna().astype(str).unique() if "_kaynak_excel_hash" in ev.columns else []
    kaynak_uyumlu = not len(eski_hashler) or KAYNAK_EXCEL_HASHI in eski_hashler
    if not kaynak_uyumlu:
        print("⚠️ Doğrulama dosyası farklı Excel'e ait; metrik hesabı atlandı.")
    etiketli=ev[ev["_manuel_etiket"].isin(["temiz","hatali","sil"])].copy()
    if kaynak_uyumlu and len(etiketli)>10:
        ds2 = pd.DataFrame()
        gecerli_hashler = etiketli["_kayit_hash"].dropna().astype(str) if "_kayit_hash" in etiketli.columns else pd.Series(dtype=str)
        gecerli_hashler = gecerli_hashler[gecerli_hashler.str.len() > 0]
        if len(gecerli_hashler) > 0:
            veri["_tmp_hash"] = veri["_kayit_metni"].apply(lambda x: hashlib.sha256(tekrar_icin_normalize(x).encode("utf-8")).hexdigest() if x else "")
            etiket_haritasi = (etiketli.drop_duplicates("_kayit_hash", keep="last").set_index("_kayit_hash")["_manuel_etiket"])
            ds2 = veri[veri["_tmp_hash"].isin(etiket_haritasi.index)].copy()
            ds2["_manuel_etiket"] = ds2["_tmp_hash"].map(etiket_haritasi)
            veri.drop(columns=["_tmp_hash"], inplace=True)
        elif "_veri_indeksi" in etiketli.columns:
            indeksler = etiketli["_veri_indeksi"].dropna().astype(int)
            indeksler = indeksler[indeksler.isin(veri.index)]
            ds2 = veri.loc[indeksler].copy()
            etiket_haritasi = (etiketli.dropna(subset=["_veri_indeksi"]).assign(_veri_indeksi=lambda x: x["_veri_indeksi"].astype(int)).set_index("_veri_indeksi")["_manuel_etiket"])
            ds2["_manuel_etiket"] = ds2.index.map(etiket_haritasi)
        else:
            print("⚠️ Doğrulama eşleştirme alanı bulunamadı.")
        if "_manuel_etiket" in ds2.columns:
            ds2 = ds2[ds2["_manuel_etiket"].isin(["temiz","hatali","sil"])]

        if len(ds2)>10:
            from sklearn.metrics import classification_report, confusion_matrix
            yt=ds2["_manuel_etiket"].apply(lambda x:1 if x in["sil","hatali"]else 0)
            yp=ds2["_oneri_aksiyon"].apply(lambda x:0 if x=="TEMİZ"else 1)
            print("="*50); print("📊 DOĞRULAMA SETİ"); print("="*50)
            print(f"Etiketli: {len(ds2)}")
            print(f"\n{confusion_matrix(yt,yp,labels=[0,1])}")
            print(f"\n{classification_report(yt,yp,target_names=['TEMİZ','HATALI'])}")
            fp=ds2[(yt==0)&(yp==1)]; fn=ds2[(yt==1)&(yp==0)]
            if len(fp)>0: print(f"\n⚠️ Yanlış pozitif: {len(fp)}")
            if len(fn)>0: print(f"⚠️ Yanlış negatif: {len(fn)}")
        else: print("⚠️ Yetersiz etiket.")
    elif kaynak_uyumlu: print("⚠️ Etiketli satır yok.")
else: print(f"⚠️ Doğrulama seti bulunamadı: {dogrulama_yolu}")
print("✅ Hücre 13 tamam.")

## 📤 Hücre 14: Çıktılar + Grafikler

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"]=100

orijinal_kolonlar = [k for k in veri.columns if not k.startswith("_") and "embed" not in k.lower()]
korunan_veri = veri[veri["_oneri_aksiyon"]!="OTOMATİK_SİL"].copy()
kk = [k for k in orijinal_kolonlar if k in korunan_veri.columns]
ky = os.path.join(CIKTI_DIZINI, "korunan_veri.xlsx"); korunan_veri[kk].to_excel(ky, index=False, engine="openpyxl")
print(f"✅ Korunan: {ky} ({len(korunan_veri)})")
sil_veri = veri[veri["_oneri_aksiyon"]=="OTOMATİK_SİL"].copy()
sy = os.path.join(CIKTI_DIZINI, "otomatik_silinecekler.xlsx"); sil_veri[kk].to_excel(sy, index=False, engine="openpyxl")
print(f"✅ Silinecek: {sy} ({len(sil_veri)})")
rk = [k for k in orijinal_kolonlar if k in veri.columns]
ek = ["_hata_siniflari","_oneri_aksiyon","_benzer_seviye","_benzer_skor","_benzer_madde_no","_benzer_metin",
      "_tam_tekrar_indeksi","_outlier_dbscan","_outlier_iso","_dil","_yazim_orani","_token_sayisi","_ttr","_kategori_cosine"]
ek = [k for k in ek if k in veri.columns]
rv = veri[rk+ek].copy(); rv.columns = [c.replace("_"," ").strip().title() for c in rv.columns]
ry = os.path.join(CIKTI_DIZINI, "tum_kayitlar_raporu.xlsx"); rv.to_excel(ry, index=False, engine="openpyxl")
print(f"✅ Rapor: {ry} ({len(rv)})")

ozet = {"olusturma":datetime.now().isoformat(),"kaynak":os.path.basename(EXCEL_DOSYASI),
         "toplam":int(len(veri)),"temiz":int(temiz_satir),"hatali":int(hatali_satir),
         "aksiyon":{k:int(v) for k,v in aksiyon_dagilimi.items()},"hatalar":{},"model":MODEL_ADI,"boyut":int(boyut),"surum":"3.1"}
for k,(et,_) in HATA_SINIFI_ETIKETLERI.items():
    s=int(veri[k].sum()) if k in veri.columns else 0
    if s>0: ozet["hatalar"][et]=s
with open(os.path.join(CIKTI_DIZINI,"ozet.json"),"w",encoding="utf-8") as f: json.dump(ozet,f,ensure_ascii=False,indent=2)

aksiyon_renkler = {"TEMİZ":"#2ecc71","İNCELE":"#f39c12","GÜÇLÜ_ADAY":"#e74c3c","OTOMATİK_SİL":"#c0392b"}
ad_veri = aksiyon_dagilimi.to_dict(); etiketler=list(ad_veri.keys()); degerler=list(ad_veri.values())
renkler=[aksiyon_renkler.get(e,"#95a5a6") for e in etiketler]
def pasta_yuzdesi(yuzde): return f"{yuzde:.1f}%" if yuzde >= 1 else ""
fig1,ax1=plt.subplots(figsize=(8,6))
dilimler,_,_=ax1.pie(degerler,labels=None,autopct=pasta_yuzdesi,colors=renkler,startangle=90)
ax1.legend(dilimler,etiketler,loc="center left",bbox_to_anchor=(1,0.5))
ax1.set_title(f"Aksiyon Dağılımı (n={len(veri)})")
g1y=os.path.join(CIKTI_DIZINI,"aksiyon_dagilimi.png"); plt.savefig(g1y,dpi=150,bbox_inches="tight"); plt.show()
print(f"✅ Grafik: {g1y}")
if len(benzerlik_skorlari)>1:
    fig2,ax2=plt.subplots(figsize=(10,5))
    ax2.hist(benzerlik_skorlari,bins=40,color="#3498db",edgecolor="white",alpha=0.8)
    ax2.axvline(GA,color="red",linestyle="--",label=f"Güçlü aday ({GA})")
    ax2.axvline(IE,color="orange",linestyle="--",label=f"İncele ({IE})")
    ax2.set_xlabel("Cosine Skoru"); ax2.set_ylabel("Frekans")
    ax2.set_title(f"Benzerlik Skoru Dağılımı (n={len(benzerlik_skorlari)})"); ax2.legend(fontsize=8)
    g2y=os.path.join(CIKTI_DIZINI,"benzerlik_histogrami.png"); plt.savefig(g2y,dpi=150,bbox_inches="tight"); plt.show()
    print(f"✅ Grafik: {g2y}")

print(f"\n{'='*50}\n📁 {CIKTI_DIZINI}/")
print(f"   📄 korunan_veri.xlsx ({len(korunan_veri)}) | otomatik_silinecekler.xlsx ({len(sil_veri)}) | tum_kayitlar_raporu.xlsx ({len(rv)})")
print(f"{'='*50}")
print(f"\n✅ DOĞRULAMA: {len(veri)} = {len(korunan_veri)} + {len(sil_veri)} → {'✅ EŞİT' if len(veri)==len(korunan_veri)+len(sil_veri) else '❌ EŞİT DEĞİL!'}")
print(f"\n🔖 Doğrulama: {dogrulama_yolu}\n💾 Checkpoint: {CKPT_DIZINI}/")
print(f"\n🎉 İŞLEM TAMAMLANDI!")
print(f"   python excel_ice_aktar.py {ky}")
print(f"   python vektorleri_olustur.py")

---
## v3.1 Final
- 🔴 Standart `_konu` KeyError giderildi · 🔴 Model+Config+Excel hash checkpoint · Hunspell opsiyonel
- 🟠 Semantic search top_k genişletildi · Doğrulama indeks+hash çift koruma
- 🟡 Colab'da pandas/numpy/matplotlib sabitlenmez
---
**HadesBot · v3.1 · 2026**